In [1]:
from dotenv import load_dotenv
import getpass, os

load_dotenv()

True

In [2]:
if "GROQ_API_KEY" not in os.environ:
    os.environ["GROQ_API_KEY"] = getpass.getpass("Enter GROQ API key")
    
if "TAVILY_API_KEY" not in os.environ:
    os.environ["TAVILY_API_KEY"] = getpass.getpass("Enter Tavily API key")
    
groq_api_key = os.getenv("GROQ_API_KEY")


In [24]:
from langchain_groq import ChatGroq
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy
from langchain_tavily import TavilySearch
from langchain.tools import tool
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from pydantic import BaseModel, Field
from bs4 import BeautifulSoup
import requests, trafilatura, re
from readability import Document

In [4]:
llm = ChatGroq(
    model="openai/gpt-oss-safeguard-20b"
)

In [5]:
# initiate tavily
tavily = TavilySearch(
    max_results=5,
    topic="general"
)

In [6]:
# Response Format
class Answer(BaseModel):
    results: str = Field(description="Results of the search")
    urls: list[str] = Field(description="List of all urls(sources) returned from the results")

## Search Agent

In [7]:
search_agent = create_agent(
    model=llm,
    tools=[tavily],
    system_prompt="You are a professional researcher. Given a topic, traverse the internet and return the most factual sources. The information should be in string format(No Markdown). It should contain the results, and the url of each result. The urls should be structured such that they can be extracted later",
    # response_format=ToolStrategy(Answer)
)

In [8]:
input = {
    "messages": [
        {
            "role": "user",
            "content": "Is sifuna more popular than Ruto in Kenya's 2026 political scene?",
        }
    ]
}

In [9]:
results = search_agent.invoke(input)

In [10]:
search_results = results['messages'][-1].content

In [11]:
# search_results = results['structured_response']

## Reader Agent

In [12]:

@tool
def scrape_url(url: str):
    """
    Scrape and extract clean readable content from a URL
    Uses multiple extraction strategies for better reliability
    """
    
    try:
        response = requests.get(url, timeout=15)
        response.raise_for_status()
        
        html = response.text
        
        #_________________________________
        # Strategy 1 - trafilatura (Best for articles/blogs)
        #__________________________________
        
        extracted = trafilatura.extract(
            html, 
            output_format="json", 
            with_metadata=True, 
            include_comments=False, 
            include_tables=False
            )
        
        if extracted and len(extracted.strip()) > 200:
            cleaned = re.sub(r'\s+', ' ', extracted)
            return cleaned[:5000]
        
        
        #______________________________________
        # Strategy 2 - Readability
        #____________________________________
        doc = Document(html)
        clean_html = doc.summary()
        
        soup = BeautifulSoup(clean_html, "html.parser")
        
        # strip these tags and return the remaining
        for tag in soup([
            "script",
            "style",
            "nav",
            "footer",
            "header",
            "aside",
            "form"
        ]):
            tag.decompose()
            
        text = soup.get_text(separator=" " , strip=True)
        
        if text and len(text.strip()) > 200:
            cleaned = re.sub(r'\s+', ' ', text)
            return cleaned[:5000]
        
        #_______________________________________
        # Strategy 3 - Fallback full page extraction
        #________________________________________
        soup = BeautifulSoup(html, "html.parser")
            
        for tag in soup([
            "script",
            "style",
            "nav",
            "footer",
            "header",
            "aside",
            "form"
        ]):
            tag.decompose()
            
        text = soup.get_text(separator=" " , strip=True)
        
        cleaned = re.sub(r'\s+', ' ', text)
        
        if cleaned:
            return cleaned[:5000]
        
        return "Could not extract meaningful content from the page"
    
    except requests.exceptions.Timeout:
        return "Request timed out while scraping the URL"
    
    except requests.exceptions.HTTPError as e:
        return f"HTTP error occurred: {str(e)}"
    except Exception as e:
        return f"Could not scrape URL: {str(e)}"

In [13]:
class Results(BaseModel):
    report: str

In [14]:

reader_agent = create_agent(
    model=llm,
    tools=[scrape_url],
    system_prompt=(
        "You are a web research reader. You MUST call scrape_url before writing your answer. "
        "Choose one URL from the supplied URLs, call scrape_url with that exact URL, and base "
        "your report on the returned page content. Never answer from the search results alone. "
        "After the tool returns, provide the final report as plain text."
    ),
    # response_format=ToolStrategy(Results)
)

In [15]:

input = {
    "messages": [
        {
            "role": "user",
            "content": (
                "Select the single best source URL below. You must call scrape_url with that URL "
                "before producing the report. Then write a detailed plain-text report based on the "
                "scraped page content.\n\n"
                f"Search Results: {search_results}"
            ),
        }
    ]
}

In [16]:
scraped_content = reader_agent.invoke(input)

In [22]:
response = scraped_content['messages'][-1].content

## Writer Agent

In [28]:
combined_report = (
    f"Search Results: {search_results}"
    f"Scraped Content: {response}"
)

In [25]:
writer_prompt = ChatPromptTemplate(
    [
        (
            "system",
            "You are a skilled research assistant. Given a report, form a detailed formal report about it."
            """The report should include the sources, detailed explanations and overview""",
        ),
        (
            "human",
            """Write a detailed research report on the topic below.
        
                    Research Gathered:
                    {research}
        
                    Structure the report as:
                    - Introduction
                    - Key Findings (minimum 3 well-explained points)
                    - Conclusion
                    - Sources (list all URLs found in the research)
        
                    Be detailed, factual and professional.
                    """,
        ),
    ]
)

In [26]:
writer_chain = writer_prompt | llm | StrOutputParser()

In [29]:
final_draft = writer_chain.invoke({"research": combined_report})

In [32]:
final_draft

'**Research Report**  \n**Title:** *Popularity of Sifuna versus President William\u202fRuto in Kenya’s 2026 Political Landscape*  \n\n**Prepared by:** Research Assistant (2026)  \n**Date:** 7\u202fSeptember\u202f2026  \n\n---\n\n### 1. Introduction  \n\nThe 2027 Kenyan presidential election is rapidly approaching, and the political arena is dominated by a handful of high‑profile contenders. Among them, President William\u202fRuto (the incumbent) and former Deputy President Gideon “Sifuna” Kinyua have attracted the most media attention and public scrutiny. Understanding their relative popularity is essential for predicting electoral outcomes, coalition dynamics, and the broader direction of Kenya’s political future.\n\nThis report synthesises the most recent, nationally representative opinion polls conducted between May and July\u202f2026. The data come from two independent polling agencies—**TIFA Research** and **Infotrak**—and are corroborated by media outlets that reported on these s

## Critic Chain

In [33]:
critic_prompt = ChatPromptTemplate.from_messages([
     ("system", "You are a sharp and constructive research critic. Be honest and specific."),
    ("human", """Review the research report below and evaluate it strictly.

        Report:
        {report}

        Respond in this exact format:

        Score: X/10

        Strengths:
        - ...
        - ...

        Areas to Improve:
        - ...
        - ...

        One line verdict:
        ..."""
        ),
])

critic_chain = critic_prompt | llm | StrOutputParser()

In [34]:
critic = critic_chain.invoke({"report": final_draft})

In [35]:
critic

'Score: 7/10\n\n**Strengths:**\n- **Clear structure**: The report follows a logical flow—introduction, key findings, contextual analysis, conclusion—making it easy to follow.\n- **Data transparency**: Poll results are tabulated with specific percentages, dates, and agency names, allowing readers to see the raw figures.\n- **Multiple sources**: Use of two independent polling agencies (TIFA and Infotrak) and corroborating media reports enhances credibility.\n- **Contextual framing**: The discussion of methodology, regional representation, and political implications adds depth beyond mere numbers.\n\n**Areas to Improve:**\n- **Title–content alignment**: The title references “2026 political landscape” while the narrative centers on the 2027 election; clarify the temporal focus.\n- **Methodological detail**: Provide explicit sample sizes, weighting procedures, question wording, and margin‑of‑error calculations for each poll to allow critical assessment of validity.\n- **Consistency of data*